# Étape 6 — La démo interactive

Une interface web pour montrer le modèle en marche, avec un lien public à
partager pendant la soutenance.

**Ce qu'elle a de particulier : elle s'audite elle-même.** Chaque image produite
est comparée aux 29 images d'entraînement, et le résultat est placé sur
l'échelle établie à l'Étape 5 — le seuil de recopie mesuré sur les photos que le
modèle n'a jamais vues. Montrer un modèle génératif sans dire s'il recopie
serait incomplet ; ici la réponse s'affiche à chaque génération.

**Durée : environ 6 minutes** avant d'avoir le lien. Il faut un GPU T4 et
`runs_r8.zip`.

> **Exécution → Modifier le type d'exécution → T4 GPU** avant de commencer.

## 1. Environnement

In [ ]:
# torchao 0.10 est preinstalle sur Colab et peft exige >= 0.16 :
# sans cette desinstallation, le branchement de la LoRA echoue.
!pip -q uninstall -y torchao 2>&1 | tail -1

!pip -q install 'diffusers>=0.27' 'peft>=0.10' 'transformers>=4.38' \
    accelerate safetensors gradio pyyaml 2>&1 | tail -2

import torch, gradio
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
print('gradio', gradio.__version__)

## 2. Dépôt et jeu de données

Les 29 images d'entraînement sont nécessaires : ce sont elles que l'audit
interroge pour trouver le plus proche voisin de chaque image produite.

In [ ]:
!git clone -q https://github.com/HatimTj/lora-zellige-patrimoine.git /content/projet 2>/dev/null || (cd /content/projet && git fetch -q && git reset -q --hard origin/main)
%cd /content/projet
!python src/collect_wikimedia.py --reconstruire \
  && python src/prepare_data.py \
  && python src/legender.py

In [ ]:
from pathlib import Path

n_train = len(list(Path('data/processed/train').glob('*.jpg')))
print(f'{n_train} images d entrainement')
assert n_train > 20, "jeu d entrainement incomplet : l'audit de recopie serait fausse"

seuil = Path('results/surapprentissage.json')
print('seuil de recopie mesure :', 'present' if seuil.exists() else 'ABSENT (repli 0.908)')

## 3. Récupérer la LoRA

Glisse **`runs_r8.zip`** dans le panneau Fichiers (icône dossier à gauche), puis
exécute la cellule.

In [ ]:
from pathlib import Path
import shutil, zipfile

FINAL = Path('runs/r8/pytorch_lora_weights.safetensors')
Path('runs/r8').mkdir(parents=True, exist_ok=True)

def installer(archive):
    zipfile.ZipFile(archive).extractall('.')
    print('decompresse :', archive)

if FINAL.exists():
    print('poids deja presents')
else:
    trouvees = sorted(Path('/content').glob('*.zip')) + sorted(Path('.').glob('*.zip'))
    if trouvees:
        installer(trouvees[0])
    else:
        print('aucune archive trouvee, ouverture du televersement...')
        from google.colab import files
        for nom in files.upload():
            if nom.endswith('.zip'):
                installer(nom)
            elif nom.endswith('.safetensors'):
                shutil.move(nom, FINAL)

assert FINAL.exists(), 'poids de la LoRA introuvables'
print(f'{FINAL}  ({FINAL.stat().st_size/1e6:.1f} Mo)')

## 4. Vérification avant lancement

L'interface se construit sans charger le modèle. Si une incompatibilité de
version de Gradio existe, elle apparaît ici en deux secondes plutôt qu'après la
minute de chargement du pipeline.

In [ ]:
import sys
sys.path.insert(0, 'demo')
import app

print('gradio majeur detecte :', app.VERSION_GRADIO)
essai = app.construire('runs/r8')
print('interface construite :', len(essai.get_config_file()['components']), 'composants')
del essai

## 5. Lancer la démo

La cellule affiche deux liens. Celui en `.gradio.live` est **public** : c'est
celui à partager ou à ouvrir sur un téléphone pendant la soutenance. Il reste
valable 72 heures.

La cellule ne se termine pas — c'est normal, le serveur tourne. Pour l'arrêter,
clique sur le carré d'arrêt.

Premier lancement : environ une minute de chargement (Stable Diffusion, la LoRA,
puis DINOv2 qui vectorise les 29 images d'entraînement).

In [ ]:
!python demo/app.py --partager

## 6. La vidéo — 60 à 90 secondes

Enregistre l'écran pendant que tu manipules la démo. Un déroulé qui tient dans
le temps imparti et qui montre les trois choses qu'un jury attend :

| temps | ce que tu montres | ce que tu dis |
|---|---|---|
| 0:00-0:10 | l'interface, le prompt D1 déjà rempli | le projet en une phrase : 29 photos, 6 Mo, un style |
| 0:10-0:25 | **Générer** — les deux images arrivent | à gauche le modèle d'origine, à droite le même modèle avec 6 Mo en plus |
| 0:25-0:45 | curseur sur 0,25 puis 0,75 puis 1,25 | le style ne monte pas en ligne droite : il bascule vers 1,0, du carreau peint à la mosaïque assemblée |
| 0:45-1:05 | le panneau d'audit, puis le voisin le plus proche | la ressemblance est de 0,84, sous le seuil de 0,908 mesuré sur des photos jamais vues : même composition, autre palette, autres tesselles |
| 1:05-1:20 | décocher `zlj_style`, prompt du chien | sans le mot déclencheur le modèle est intact : la LoRA n'a rien écrasé |
| 1:20-1:30 | les licences dans le volet du bas | données CC, traçables une par une |

Astuce : sur un prompt déjà généré, bouger le curseur ne recalcule que l'image
de droite — la réponse arrive deux fois plus vite, ce qui rend la démonstration
du curseur fluide à l'écran.